# Notebook 07 — MIMIC-III Kalibrasyon Düzeltmesi
## Tamamen Bağımsız | Hiçbir Eski Notebook Gerekmez

**Tek ön koşul:** Google Drive'ında `HF_Revisit_Project/` klasörünün olması ve PhysioNet hesabının BigQuery'e bağlı olması.

Bu notebook:
1. Kaggle verisini yükler ve RF modelini **sıfırdan eğitir**
2. BigQuery ile MIMIC-III kohortunu **kendisi oluşturur** (NB04 ile aynı sorgular)
3. Modeli MIMIC verisine uygular, tahminleri üretir
4. Hold-out kalibrasyon düzeltmesi yapar (Platt + İzotonik)
5. Sonuçları kaydeder — CSV + PNG + run log


## Hücre 1 — Kurulum ve Kütüphaneler

In [ ]:

import subprocess, sys, os, re, json, warnings, datetime
from pathlib import Path

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['pandas-gbq', 'google-cloud-bigquery', 'db-dtypes', 'scikit-learn']:
    pip_install(pkg)

import numpy  as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import chi2
from sklearn.ensemble     import RandomForestClassifier
from sklearn.pipeline     import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model    import LogisticRegression
from sklearn.isotonic        import IsotonicRegression
from sklearn.calibration      import calibration_curve
from sklearn.metrics import roc_auc_score, brier_score_loss
from google.cloud import bigquery
import pandas_gbq

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Kutuphaneler hazir.')


Kutuphaneler hazir.


## Hücre 2 — Drive Mount & Proje Yolları

In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = Path('/content/drive/MyDrive/HF_Revisit_Project')
DATA_DIR     = PROJECT_ROOT / 'data'
RESULTS_DIR  = PROJECT_ROOT / 'results'
FIGURES_DIR  = RESULTS_DIR  / 'figures'
MODELS_DIR   = PROJECT_ROOT / 'models'
LOGS_DIR     = PROJECT_ROOT / 'logs'

for d in [RESULTS_DIR, FIGURES_DIR, MODELS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── BigQuery proje ID'nizi buraya yazin ──────────────────
# PhysioNet MIMIC-III erisimi icin BigQuery'de yetkilendirilmis olmalisiniz.
GCP_PROJECT_ID = 'physionet-data-491418'   # <-- KENDI GCP ID'NIZI YAZIN
MIMIC_DATASET  = 'physionet-data.mimiciii_clinical'
MIMIC_NOTES    = 'physionet-data.mimiciii_notes'

print('Drive baglandi:', PROJECT_ROOT)
print('GCP_PROJECT_ID:', GCP_PROJECT_ID)


Mounted at /content/drive
Drive baglandi: /content/drive/MyDrive/HF_Revisit_Project
GCP_PROJECT_ID: physionet-data-491418


## Hücre 3 — BigQuery Kimlik Doğrulama

In [ ]:

from google.colab import auth as colab_auth
colab_auth.authenticate_user()

client = bigquery.Client(project=GCP_PROJECT_ID)

def bq_query(sql, desc=''):
    print('Sorgu:', desc, '...')
    t0 = datetime.datetime.now()
    df = pandas_gbq.read_gbq(sql, project_id=GCP_PROJECT_ID,
                              progress_bar_type=None, location='US')
    print('  OK:', len(df), 'satir,', (datetime.datetime.now()-t0).seconds, 'sn')
    return df

print('BigQuery hazir.')


BigQuery hazir.


## Hücre 4 — Kaggle Verisini Yükle & RF Modelini Eğit

In [ ]:

KAGGLE_PATH = DATA_DIR / 'heart_failure_clinical_records_dataset.csv'

if not KAGGLE_PATH.exists():
    raise FileNotFoundError(
        str(KAGGLE_PATH) + ' bulunamadi.\n'
        'Kaggle CSV dosyasini Drive/HF_Revisit_Project/data/ altina yukleyin.'
    )

df_kaggle = pd.read_csv(KAGGLE_PATH)
df_kaggle.columns = [c.lower() for c in df_kaggle.columns]

FEATURES_S2 = [
    'age', 'anaemia', 'creatinine_phosphokinase', 'diabetes',
    'ejection_fraction', 'high_blood_pressure', 'platelets',
    'serum_creatinine', 'serum_sodium', 'sex', 'smoking'
]
TARGET = 'death_event'

X_kg = df_kaggle[FEATURES_S2].values
y_kg = df_kaggle[TARGET].astype(int).values
print('Kaggle:', len(df_kaggle), 'hasta | Mortalite: %' + str(round(y_kg.mean()*100,1)))

# RF modelini tum Kaggle verisiyle egit (NB01 ile ayni hiperparametreler)
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=500, max_depth=6, min_samples_leaf=3,
        class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1
    ))
])
rf_pipeline.fit(X_kg, y_kg)

EF_MIN, EF_MAX = 5, 85
CPK_MEDIAN_KAGGLE = 250.0

print('Model egitildi.')


Kaggle: 299 hasta | Mortalite: %32.1
Model egitildi.


## Hücre 5 — MIMIC-III Kohort İnşası (~5-10 dk)

In [ ]:

DM_ICD9_CODES = [
    '25000','25001','25002','25003','25010','25011','25012','25013',
    '25020','25021','25022','25023','25030','25031','25032','25033',
    '25040','25041','25042','25043','25050','25051','25052','25053',
    '25060','25061','25062','25063','25070','25071','25072','25073',
    '25080','25081','25082','25083','25090','25091','25092','25093'
]
SMOKE_ICD9 = ['3051','V1582','30511','30519']

MD  = MIMIC_DATASET
MN  = MIMIC_NOTES

# SORGU 1: Temel kohort + mortalite
SQL_BASE = '''
WITH
hf_first AS (
  SELECT d.SUBJECT_ID, MIN(d.HADM_ID) AS HADM_ID
  FROM `{MD}.diagnoses_icd` d
  WHERE d.ICD9_CODE LIKE "428%"
  GROUP BY d.SUBJECT_ID
),
ages AS (
  SELECT fa.SUBJECT_ID, fa.HADM_ID,
    DATE_DIFF(DATE(a.ADMITTIME), DATE(p.DOB), YEAR) AS age_raw
  FROM hf_first fa
  JOIN `{MD}.admissions` a ON fa.HADM_ID = a.HADM_ID
  JOIN `{MD}.patients`   p ON fa.SUBJECT_ID = p.SUBJECT_ID
)
SELECT fa.SUBJECT_ID, fa.HADM_ID,
  LEAST(GREATEST(ag.age_raw, 18), 89)        AS age,
  CASE WHEN p.GENDER="M" THEN 1 ELSE 0 END   AS sex,
  a.HOSPITAL_EXPIRE_FLAG                      AS DEATH_EVENT
FROM hf_first fa
JOIN `{MD}.admissions` a ON fa.HADM_ID  = a.HADM_ID
JOIN `{MD}.patients`   p ON fa.SUBJECT_ID = p.SUBJECT_ID
JOIN ages ag               ON fa.HADM_ID  = ag.HADM_ID
WHERE ag.age_raw >= 18
'''.replace('{MD}', MD)
df_base = bq_query(SQL_BASE, 'Temel HF kohortu')

# SORGU 2: Lab degerleri (ilk 24 saat)
SQL_LABS = '''
WITH
hf_first AS (
  SELECT d.SUBJECT_ID, MIN(d.HADM_ID) AS HADM_ID
  FROM `{MD}.diagnoses_icd` d
  WHERE d.ICD9_CODE LIKE "428%"
  GROUP BY d.SUBJECT_ID
),
labs_raw AS (
  SELECT l.HADM_ID, l.ITEMID, l.VALUENUM,
    ROW_NUMBER() OVER (PARTITION BY l.HADM_ID, l.ITEMID ORDER BY l.CHARTTIME) AS rn
  FROM `{MD}.labevents` l
  JOIN hf_first hf ON l.HADM_ID = hf.HADM_ID
  JOIN `{MD}.admissions` a ON l.HADM_ID = a.HADM_ID
  WHERE l.ITEMID IN (50912,50983,50824,51265,50910,50911,51221,51222)
    AND l.VALUENUM IS NOT NULL AND l.VALUENUM > 0
    AND l.CHARTTIME BETWEEN a.ADMITTIME
      AND TIMESTAMP_ADD(a.ADMITTIME, INTERVAL 24 HOUR)
)
SELECT HADM_ID,
  MAX(CASE WHEN ITEMID=50912 AND rn=1 THEN VALUENUM END) AS creatinine,
  MAX(CASE WHEN ITEMID IN (50983,50824) AND rn=1 THEN VALUENUM END) AS sodium,
  MAX(CASE WHEN ITEMID=51265 AND rn=1 THEN VALUENUM END) AS platelets,
  MAX(CASE WHEN ITEMID IN (50910,50911) AND rn=1 THEN VALUENUM END) AS creatinine_phosphokinase,
  MAX(CASE WHEN ITEMID=51221 AND rn=1 THEN VALUENUM END) AS hematocrit,
  MAX(CASE WHEN ITEMID=51222 AND rn=1 THEN VALUENUM END) AS hemoglobin
FROM labs_raw GROUP BY HADM_ID
'''.replace('{MD}', MD)
df_labs = bq_query(SQL_LABS, 'Lab degerleri (24h)')

# SORGU 3: Komorbiditeler
dm_str  = ','.join(['"' + c + '"' for c in DM_ICD9_CODES])
smk_str = ','.join(['"' + c + '"' for c in SMOKE_ICD9])
SQL_COM = ('''
WITH
hf_first AS (
  SELECT d.SUBJECT_ID, MIN(d.HADM_ID) AS HADM_ID
  FROM `{MD}.diagnoses_icd` d
  WHERE d.ICD9_CODE LIKE "428%"
  GROUP BY d.SUBJECT_ID
)
SELECT hf.SUBJECT_ID, hf.HADM_ID,
  MAX(CASE WHEN SUBSTR(d.ICD9_CODE,1,3)="250" OR d.ICD9_CODE IN ({DM})
      THEN 1 ELSE 0 END) AS diabetes,
  MAX(CASE WHEN SUBSTR(d.ICD9_CODE,1,3) IN ("401","402","403","404","405")
      THEN 1 ELSE 0 END) AS high_blood_pressure,
  MAX(CASE WHEN d.ICD9_CODE IN ({SMK}) OR SUBSTR(d.ICD9_CODE,1,4)="3051"
      THEN 1 ELSE 0 END) AS smoking
FROM hf_first hf
LEFT JOIN `{MD}.diagnoses_icd` d ON hf.HADM_ID = d.HADM_ID
GROUP BY hf.SUBJECT_ID, hf.HADM_ID
''').replace('{MD}', MD).replace('{DM}', dm_str).replace('{SMK}', smk_str)
df_com = bq_query(SQL_COM, 'Komorbiditeler')

# SORGU 4: Echo notlarindan EF
SQL_ECHO = '''
WITH
hf_first AS (
  SELECT d.SUBJECT_ID, MIN(d.HADM_ID) AS HADM_ID
  FROM `{MD}.diagnoses_icd` d
  WHERE d.ICD9_CODE LIKE "428%"
  GROUP BY d.SUBJECT_ID
)
SELECT n.SUBJECT_ID, hf.HADM_ID,
  SUBSTR(n.TEXT, 1, 3000) AS note_text
FROM `{MN}.noteevents` n
JOIN hf_first hf ON n.SUBJECT_ID = hf.SUBJECT_ID
WHERE n.CATEGORY = "Echo"
  AND (LOWER(n.TEXT) LIKE "%ejection fraction%"
       OR REGEXP_CONTAINS(LOWER(n.TEXT), r"\\bef\\b")
       OR REGEXP_CONTAINS(LOWER(n.TEXT), r"lvef"))
QUALIFY ROW_NUMBER() OVER (PARTITION BY n.SUBJECT_ID ORDER BY n.CHARTDATE) = 1
'''.replace('{MD}', MD).replace('{MN}', MN)
df_echo = bq_query(SQL_ECHO, 'Echo notlari')
print('4 sorgu tamamlandi.')


Sorgu: Temel HF kohortu ...
  OK: 10139 satir, 1 sn
Sorgu: Lab degerleri (24h) ...
  OK: 9935 satir, 2 sn
Sorgu: Komorbiditeler ...
  OK: 10154 satir, 0 sn
Sorgu: Echo notlari ...
  OK: 7218 satir, 7 sn
4 sorgu tamamlandi.


## Hücre 6 — EF Parse & Kohort Birleştirme

In [ ]:

EF_PATTERNS = [
    r'lvef[\s:=~of]*([\d]+(?:\.[\d]+)?)(?:\s*[-\u2013to]+\s*[\d]+)?\s*%?',
    r'ejection\s+fraction[\s:=~of]*([\d]+(?:\.[\d]+)?)(?:\s*[-\u2013to]+\s*[\d]+)?\s*%?',
    r'\bef[\s:=~of]+([\d]+(?:\.[\d]+)?)(?:\s*[-\u2013to]+\s*[\d]+)?\s*%?',
    r'([\d]+(?:\.[\d]+)?)\s*%\s*(?:ef|ejection|lvef)',
]
CATEGORICAL_EF = {
    r'severely\s+reduced|severely\s+depressed': 15.0,
    r'moderately\s+(?:to\s+severely\s+)?reduced': 25.0,
    r'moderately\s+reduced|moderately\s+depressed': 30.0,
    r'mildly\s+(?:to\s+moderately\s+)?reduced': 40.0,
    r'mildly\s+reduced|mildly\s+depressed': 45.0,
    r'low\s+normal|low-normal': 50.0,
    r'(?:preserved|normal)\s+(?:ef|function)': 60.0,
    r'hyperdynamic': 70.0,
}

def parse_ef(text):
    if not isinstance(text, str): return None
    t = text.lower()
    for pat in EF_PATTERNS:
        m = re.search(pat, t)
        if m:
            try:
                v = float(m.group(1))
                if EF_MIN <= v <= EF_MAX: return v
            except: pass
    rng = r'(?:lvef|ef|ejection\s+fraction)[\s:=~of]*([\d]+)\s*[-\u2013to]+\s*([\d]+)\s*%?'
    m = re.search(rng, t)
    if m:
        try:
            mid = (float(m.group(1)) + float(m.group(2))) / 2
            if EF_MIN <= mid <= EF_MAX: return mid
        except: pass
    for cp, cv in CATEGORICAL_EF.items():
        if re.search(cp, t): return cv
    return None

df_echo['ejection_fraction'] = df_echo['note_text'].apply(parse_ef)
df_ef = df_echo.dropna(subset=['ejection_fraction'])[['SUBJECT_ID','HADM_ID','ejection_fraction']]
print('EF parse:', len(df_ef), 'hasta | Medyan EF:', round(df_ef.ejection_fraction.median(),0))

# Kohort birlestirme
df = df_base.merge(df_ef[['SUBJECT_ID','ejection_fraction']], on='SUBJECT_ID', how='inner')
print('EF merge:', len(df))
df = df.merge(df_labs, on='HADM_ID', how='left')
df = df.dropna(subset=['creatinine','sodium','platelets'])
print('Lab filtresi:', len(df))
df = df.merge(df_com[['SUBJECT_ID','diabetes','high_blood_pressure','smoking']],
              on='SUBJECT_ID', how='left')
for col in ['diabetes','high_blood_pressure','smoking']:
    df[col] = df[col].fillna(0).astype(int)

# Anaemi: WHO kriteri
df['anaemia'] = 0
if 'hemoglobin' in df.columns and 'hematocrit' in df.columns:
    mask_f = (df['sex']==0) & ((df['hemoglobin']<12.0) | (df['hematocrit']<36.0))
    mask_m = (df['sex']==1) & ((df['hemoglobin']<13.0) | (df['hematocrit']<39.0))
    df.loc[mask_f | mask_m, 'anaemia'] = 1

# Fizyolojik aralik filtresi
df = df[(df['ejection_fraction'].between(EF_MIN, EF_MAX)) &
        (df['creatinine'] < 20.0) &
        (df['sodium'].between(100, 180)) &
        (df['platelets'].between(5, 1500)) &
        (df['age'] >= 18)].copy()

df = df.rename(columns={'creatinine':'serum_creatinine','sodium':'serum_sodium'})
df['creatinine_phosphokinase'] = df.get('creatinine_phosphokinase',
    pd.Series(dtype=float)).fillna(CPK_MEDIAN_KAGGLE)

df_primary = df.reset_index(drop=True)
print('\nFinal kohort:', len(df_primary), 'hasta')
print('  Mortalite: %' + str(round(df_primary.DEATH_EVENT.mean()*100, 1)))
print('  Yas (medyan):', df_primary.age.median())


EF parse: 4095 hasta | Medyan EF: 40.0
EF merge: 4094
Lab filtresi: 3911

Final kohort: 3911 hasta
  Mortalite: %14.4
  Yas (medyan): 74.0


## Hücre 7 — MIMIC Tahminleri Üret & Kaydet

In [ ]:

MIMIC_COLS = [
    'age', 'anaemia', 'creatinine_phosphokinase', 'diabetes',
    'ejection_fraction', 'high_blood_pressure', 'platelets',
    'serum_creatinine', 'serum_sodium', 'sex', 'smoking'
]

X_mimic = df_primary[MIMIC_COLS].copy()
for col in X_mimic.columns:
    n_nan = X_mimic[col].isna().sum()
    if n_nan > 0:
        X_mimic[col] = X_mimic[col].fillna(X_mimic[col].median())
        print('  ' + col + ': ' + str(n_nan) + ' NaN -> medyan dolduruldu')

y_mimic = df_primary['DEATH_EVENT'].astype(int).values
y_prob  = rf_pipeline.predict_proba(X_mimic.values)[:, 1]

pred_path = RESULTS_DIR / 'NB04_predictions_mimic.csv'
pd.DataFrame({'y_true': y_mimic, 'y_prob': y_prob}).to_csv(pred_path, index=False)

print('MIMIC tahminleri: n=' + str(len(y_mimic)))
print('  AUC  :', round(roc_auc_score(y_mimic, y_prob), 3))
print('  Brier:', round(brier_score_loss(y_mimic, y_prob), 3))
print('  Kayit:', str(pred_path))


MIMIC tahminleri: n=3911
  AUC  : 0.602
  Brier: 0.303
  Kayit: /content/drive/MyDrive/HF_Revisit_Project/results/NB04_predictions_mimic.csv


## Hücre 8 — Hold-out Kalibrasyon Düzeltmesi

MIMIC ikiye bölünür: **%50 kalibrasyon / %50 test**. Kalibratör kalibrasyon setinde öğrenilir, test setinde ölçülür → **sızıntısız**.

In [ ]:

def hosmer_lemeshow(y, p, g=10):
    d = pd.DataFrame({'y':y,'p':p})
    d['bin'] = pd.qcut(d['p'], q=g, labels=False, duplicates='drop')
    t = d.groupby('bin').agg(n=('y','count'),obs=('y','sum'),exp=('p','sum'))
    t['obs_neg']=t['n']-t['obs']; t['exp_neg']=t['n']-t['exp']
    hl=(((t['obs']-t['exp'])**2/t['exp'].clip(lower=1e-6))+
        ((t['obs_neg']-t['exp_neg'])**2/t['exp_neg'].clip(lower=1e-6))).sum()
    return float(hl), float(1-chi2.cdf(hl, max(t.shape[0]-2,1)))

def ece(y, p, n_bins=10):
    bins=np.linspace(0,1,n_bins+1); e=0; n=len(y)
    for i in range(n_bins):
        m=(p>=bins[i])&(p<bins[i+1])
        if m.sum()==0: continue
        e+=m.sum()/n*abs(np.asarray(y)[m].mean()-p[m].mean())
    return float(e)

# Tum MIMIC: duzeltme oncesi
hl0, p0 = hosmer_lemeshow(y_mimic, y_prob)
print('Tum MIMIC (oncesi): AUC='+str(round(roc_auc_score(y_mimic,y_prob),3))+
      '  HL_p='+str(round(p0,4))+'  ECE='+str(round(ece(y_mimic,y_prob),4)))

# Hold-out bolme
p_cal,p_test,y_cal,y_test = train_test_split(
    y_prob, y_mimic, test_size=0.50, random_state=RANDOM_SEED, stratify=y_mimic)

eps=1e-6
def to_logit(p):
    p=np.clip(p,eps,1-eps); return np.log(p/(1-p)).reshape(-1,1)

platt = LogisticRegression().fit(to_logit(p_cal), y_cal)
p_test_platt = platt.predict_proba(to_logit(p_test))[:,1]

iso = IsotonicRegression(out_of_bounds='clip').fit(p_cal, y_cal)
p_test_iso = iso.predict(p_test)

rows=[]
for name,p in [('Ham (oncesi)',p_test),
               ('Platt olcekleme',p_test_platt),
               ('Izotonik regresyon',p_test_iso)]:
    _,hp=hosmer_lemeshow(y_test,p)
    rows.append({'Yontem':name,'HL_p':round(hp,4),
                 'ECE':round(ece(y_test,p),4),
                 'Brier':round(brier_score_loss(y_test,p),4),
                 'AUC':round(roc_auc_score(y_test,p),4)})

df_calib=pd.DataFrame(rows)
print('\nMIMIC Hold-out Kalibrasyon (kal='+str(len(y_cal))+' / test='+str(len(y_test))+')')
print('='*55)
print(df_calib.to_string(index=False))
df_calib.to_csv(RESULTS_DIR/'NB07_calibration_mimic.csv',index=False)
print('\nKayit: NB07_calibration_mimic.csv')


Tum MIMIC (oncesi): AUC=0.602  HL_p=0.0  ECE=0.4091

MIMIC Hold-out Kalibrasyon (kal=1955 / test=1956)
            Yontem   HL_p    ECE  Brier   AUC
      Ham (oncesi) 0.0000 0.4094 0.3028 0.608
   Platt olcekleme 0.0051 0.0109 0.1212 0.608
Izotonik regresyon 0.1999 0.0140 0.1218 0.604

Kayit: NB07_calibration_mimic.csv


## Hücre 9 — Kalibrasyon Eğrisi Grafiği

In [ ]:

fig, ax = plt.subplots(figsize=(7,7))
ax.plot([0,1],[0,1],'k--',lw=1.5,label='Mukemmel kalibrasyon')

configs = [('Ham (oncesi)',       p_test,       '#d62728'),
           ('Platt olcekleme',    p_test_platt, '#1f77b4'),
           ('Izotonik regresyon', p_test_iso,   '#2ca02c')]

for name,p,color in configs:
    fp,mp = calibration_curve(y_test, p, n_bins=10, strategy='quantile')
    row   = df_calib[df_calib['Yontem']==name].iloc[0]
    ax.plot(mp, fp, marker='o', lw=2, color=color,
            label=name+' (ECE='+str(row.ECE)+', HL p='+str(row.HL_p)+')')

ax.set_xlabel('Ortalama tahmin edilen olasilik', fontsize=12)
ax.set_ylabel('Gozlemlenen frekans', fontsize=12)
ax.set_title('MIMIC-III Kalibrasyon Duzeltmesi\n(Hold-out test seti)', fontsize=13)
ax.legend(loc='upper left', fontsize=10)
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()

out_png = FIGURES_DIR/'NB07_calibration_mimic.png'
plt.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Kayit:', str(out_png))


Kayit: /content/drive/MyDrive/HF_Revisit_Project/results/figures/NB07_calibration_mimic.png


## Hücre 10 — Makale Metni & Run Log

In [ ]:

raw  = df_calib.iloc[0]
best = df_calib.iloc[1:].sort_values('ECE').iloc[0]

print('-'*65)
print('MAKALEYE EKLENECEK CUMLE (Bulgular bolumu):')
print('-'*65)
print(
    'Dis dogrulama hold-out test setinde (n='+str(len(y_test))+') '
    'modelin ham tahminleri kotu kalibre idi '
    '(Hosmer-Lemeshow p = '+str(raw.HL_p)+', ECE = '+str(raw.ECE)+').\n'
    +str(best.Yontem)+' ile kalibrasyon duzeltmesi sonrasi ECE '
    +str(raw.ECE)+"'ten "+str(best.ECE)+"'e geriledi; "
    'ayrim gucu korundi (AUC '+str(raw.AUC)+' -> '+str(best.AUC)+').'
)

import json, datetime
log = {
    'notebook': 'NB07',
    'timestamp': datetime.datetime.now().isoformat(),
    'n_mimic': int(len(y_mimic)),
    'n_calib': int(len(y_cal)),
    'n_test':  int(len(y_test)),
    'before': {'AUC': round(roc_auc_score(y_mimic,y_prob),4),
               'HL_p': round(p0,4),
               'ECE': round(ece(y_mimic,y_prob),4)},
    'results': df_calib.to_dict(orient='records')
}
with open(LOGS_DIR/'NB07_run_log.json','w') as f:
    json.dump(log, f, indent=2)
print('\nNB07_run_log.json kaydedildi.')
print('\n=== NB07 TAMAMLANDI ===')


-----------------------------------------------------------------
MAKALEYE EKLENECEK CUMLE (Bulgular bolumu):
-----------------------------------------------------------------
Dis dogrulama hold-out test setinde (n=1956) modelin ham tahminleri kotu kalibre idi (Hosmer-Lemeshow p = 0.0, ECE = 0.4094).
Platt olcekleme ile kalibrasyon duzeltmesi sonrasi ECE 0.4094'ten 0.0109'e geriledi; ayrim gucu korundi (AUC 0.608 -> 0.608).

NB07_run_log.json kaydedildi.

=== NB07 TAMAMLANDI ===
